In [1]:
from pathlib import Path

import uproot

DATA_FILENAME = "JetNtuple_RunIISummer16_13TeV_MC_1.root"
SEARCH_ROOTS = (Path.cwd(), Path.cwd().parent)
ROOT_PATH = next(
    (base / "data" / "raw" / DATA_FILENAME for base in SEARCH_ROOTS
     if (base / "data" / "raw" / DATA_FILENAME).exists()),
    None,
)
if ROOT_PATH is None:
    raise FileNotFoundError(f"Could not find {DATA_FILENAME} from {Path.cwd()}")
ROOT_PATH = ROOT_PATH.resolve()
print(f"Data file: {ROOT_PATH}")

f = uproot.open(ROOT_PATH)
classnames = f.classnames(recursive=True)   # {name: ROOT classname} — digs into every subdirectory, not just the top level

# takes objects of a certain type (Tree or RNTuple)
tree_candidates = [k for k, cls in classnames.items() if "Tree" in cls or "RNTuple" in cls]

# function to extract the cycle number (version)
def cycle_num(key):
    return int(key.rsplit(";", 1)[1]) if ";" in key else 0

# the goal is to take the highest cycle (newest version)
tree_candidates.sort(key=cycle_num, reverse=True)   
tree = f[tree_candidates[0]]

print(f"Using: {tree_candidates[0]}")
#print(tree.keys())   # branch/features names

for key in tree.keys():
    print(key)

Using: AK4jets/jetTree;3
jetPt
jetEta
jetPhi
jetMass
jetGirth
jetArea
jetRawPt
jetRawMass
jetLooseID
jetTightID
jetGenMatch
jetQGl
QG_ptD
QG_axis2
QG_mult
partonFlav
hadronFlav
physFlav
isPhysUDS
isPhysG
isPhysOther
isPartonUDS
isPartonG
isPartonOther
jetChargedHadronMult
jetNeutralHadronMult
jetChargedMult
jetNeutralMult
jetMult
nPF
PF_pT
PF_dR
PF_dTheta
PF_dPhi
PF_dEta
PF_mass
PF_id
PF_fromPV
PF_fromAK4Jet
genJetPt
genJetEta
genJetPhi
genJetMass
nGenJetPF
genJetPF_pT
genJetPF_dR
genJetPF_dTheta
genJetPF_mass
genJetPF_id
eventJetMult
jetPtOrder
dPhiJetsLO
dEtaJetsLO
alpha
event
run
lumi
pthat
eventWeight
rhoAll
rhoCentral
rhoCentralNeutral
rhoCentralChargedPileUp
PV_npvsGood
Pileup_nPU
Pileup_nTrueInt


In [ ]:
# Variables to keep for the GNN setup
node_inputs = ["PF_pT", "PF_dEta", "PF_dPhi", "PF_mass", "PF_id", "PF_fromPV"]
graph_geometry = ["PF_dEta", "PF_dPhi"]
mask_vars = ["PF_fromAK4Jet"]
targets = ["isPhysUDS", "isPhysG"]
metadata = ["event"]

excluded = {
    "physFlav", "partonFlav", "hadronFlav",
    "isPhysUDS", "isPhysG", "isPhysOther",
    "isPartonUDS", "isPartonG", "isPartonOther",
}

selected_branches = [b for b in (node_inputs + mask_vars + targets + metadata) if b in tree.keys()]
missing_branches = sorted(set(node_inputs + mask_vars + targets + metadata) - set(selected_branches))

print("Selected branches:", selected_branches)
print("Missing branches:", missing_branches)

# Load only the chosen branches
data = tree.arrays(selected_branches, library="ak")